# P2-G4 Local Agent 1: Strict-Clean Data Contract Audit

이 노트북은 모델 학습을 하지 않는다. 현행 P4/P5 모델링 입력 계약이 strict-clean D08와 manual-approved feature policy를 따르는지 감사한다.

모델링 기본 입력은 `workbook/p2/p2_4/source_eda/strict_clean_v1/mart_department_model_base_2024_strict_drop.parquet`이다. 삭제된 2,650행은 복구하지 않으며, target별 표본은 `strict_target_sample_membership.csv`의 `strict_target_keep == True` 행만 사용한다.


In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 240)


def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for path in [cur, *cur.parents]:
        if (path / "workbook" / "p2" / "p2_4").exists() and (path / "scripts").exists():
            return path
    raise RuntimeError("repo root not found")


ROOT = find_repo_root(Path.cwd())
STRICT = ROOT / "workbook/p2/p2_4/source_eda/strict_clean_v1"
HANDOFF = ROOT / "workbook/p2/p2_4/source_eda/human_handoff_packet"
MODEL_INPUT = HANDOFF / "03_MODEL_INPUT"
READINESS = ROOT / "workbook/p2/p2_4/p4_modeling_readiness_v4"

STRICT_D08 = STRICT / "mart_department_model_base_2024_strict_drop.parquet"
STRICT_D08_CSV = STRICT / "mart_department_model_base_2024_strict_drop.csv"
STRICT_TARGET_MEMBERSHIP = STRICT / "strict_target_sample_membership.csv"
STRICT_TARGET_COUNTS = STRICT / "strict_target_sample_counts.csv"
MANUAL_SCOPE_POLICY = MODEL_INPUT / "manual_approved_scope_policy.json"
MANUAL_FEATURE_REGISTRY = MODEL_INPUT / "manual_approved_feature_registry.csv"
MANUAL_TARGET_POLICY = MODEL_INPUT / "manual_approved_target_and_scope_policy.csv"
MANUAL_REVIEW_REPORT = HANDOFF / "MANUAL_REVIEW_COMPLETED.md"
HUMAN_DECISIONS = STRICT / "human_required_decision_sheet_manual_completed.csv"
LEAKAGE_DECISIONS = STRICT / "target_leakage_override_sheet_manual_completed.csv"

EXPECTED_TARGET_COUNTS = {
    "a_rate_pct": 7592,
    "cd_rate_pct": 7592,
    "f_rate_pct": 7592,
    "graduate_school_progression_rate_pct": 5674,
    "health_employment_rate_pct": 5600,
    "selectivity_proxy_pct": 3119,
}

print("ROOT:", ROOT)
print("Python:", platform.python_version())
try:
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
except Exception as exc:
    commit = f"UNAVAILABLE: {exc}"
print("Git commit:", commit)


ROOT: /home/sieg/projects-wsl/SBS_dataScience
Python: 3.12.3
Git commit: 5b1a3d54266d881a839ad9a3cec750da66e94bc7


## Gate 0 - Required Strict-Clean Inputs


In [2]:
required_paths = {
    "strict_d08_parquet": STRICT_D08,
    "strict_d08_csv": STRICT_D08_CSV,
    "strict_target_sample_membership": STRICT_TARGET_MEMBERSHIP,
    "strict_target_sample_counts": STRICT_TARGET_COUNTS,
    "manual_approved_scope_policy": MANUAL_SCOPE_POLICY,
    "manual_approved_feature_registry": MANUAL_FEATURE_REGISTRY,
    "manual_approved_target_and_scope_policy": MANUAL_TARGET_POLICY,
    "manual_review_completed_report": MANUAL_REVIEW_REPORT,
    "human_required_decision_sheet": HUMAN_DECISIONS,
    "target_leakage_override_sheet": LEAKAGE_DECISIONS,
}

path_audit = pd.DataFrame(
    [
        {
            "logical_name": name,
            "path": str(path.relative_to(ROOT)),
            "exists": path.exists(),
            "size_bytes": path.stat().st_size if path.exists() else None,
        }
        for name, path in required_paths.items()
    ]
)
display(path_audit)
if not bool(path_audit["exists"].all()):
    missing = path_audit.loc[~path_audit["exists"], "path"].tolist()
    raise FileNotFoundError(f"missing required strict/manual inputs: {missing}")


,logical_name,path,exists,size_bytes
0,strict_d08_parquet,workbook/p2/p2_4/source_eda/strict_clean_v1/ma...,True,1565423
1,strict_d08_csv,workbook/p2/p2_4/source_eda/strict_clean_v1/ma...,True,11378312
2,strict_target_sample_membership,workbook/p2/p2_4/source_eda/strict_clean_v1/st...,True,3911786
3,strict_target_sample_counts,workbook/p2/p2_4/source_eda/strict_clean_v1/st...,True,440
4,manual_approved_scope_policy,workbook/p2/p2_4/source_eda/human_handoff_pack...,True,1318
5,manual_approved_feature_registry,workbook/p2/p2_4/source_eda/human_handoff_pack...,True,29026
6,manual_approved_target_and_scope_policy,workbook/p2/p2_4/source_eda/human_handoff_pack...,True,6184
7,manual_review_completed_report,workbook/p2/p2_4/source_eda/human_handoff_pack...,True,5594
8,human_required_decision_sheet,workbook/p2/p2_4/source_eda/strict_clean_v1/hu...,True,12699
9,target_leakage_override_sheet,workbook/p2/p2_4/source_eda/strict_clean_v1/ta...,True,8197


## Gate 1 - Strict D08 And Manual Policy Checks


In [3]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def as_bool(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})


d08 = pd.read_parquet(STRICT_D08)
target_counts = pd.read_csv(STRICT_TARGET_COUNTS)
target_membership = pd.read_csv(STRICT_TARGET_MEMBERSHIP)
manual_features = pd.read_csv(MANUAL_FEATURE_REGISTRY)
manual_policy = pd.read_csv(MANUAL_TARGET_POLICY)
human_decisions = pd.read_csv(HUMAN_DECISIONS)
leakage_decisions = pd.read_csv(LEAKAGE_DECISIONS)

count_map = target_counts.set_index("target_candidate")["strict_target_keep_n"].to_dict()
checks = [
    {
        "check": "strict_d08_shape",
        "actual": str(tuple(d08.shape)),
        "expected": "(7592, 151)",
        "status": "PASS" if tuple(d08.shape) == (7592, 151) else "FAIL",
    },
    {
        "check": "strict_d08_sha256",
        "actual": sha256_file(STRICT_D08),
        "expected": "current strict-clean file hash",
        "status": "INFO",
    },
    {
        "check": "manual_approved_feature_registry_shape",
        "actual": str(tuple(manual_features.shape)),
        "expected": "(198, 11)",
        "status": "PASS" if tuple(manual_features.shape) == (198, 11) else "FAIL",
    },
    {
        "check": "manual_approved_p4_use_true_n",
        "actual": int(as_bool(manual_features["manual_approved_p4_use"]).sum()),
        "expected": 131,
        "status": "PASS" if int(as_bool(manual_features["manual_approved_p4_use"]).sum()) == 131 else "FAIL",
    },
    {
        "check": "manual_target_scope_policy_rows",
        "actual": int(len(manual_policy)),
        "expected": 32,
        "status": "PASS" if len(manual_policy) == 32 else "FAIL",
    },
    {
        "check": "human_required_decision_sheet_approved",
        "actual": int(human_decisions["final_status"].astype(str).str.lower().eq("approved").sum()),
        "expected": 13,
        "status": "PASS" if int(human_decisions["final_status"].astype(str).str.lower().eq("approved").sum()) == 13 else "FAIL",
    },
    {
        "check": "target_leakage_override_sheet_keep_blocked",
        "actual": int(leakage_decisions["human_decision"].astype(str).str.lower().eq("keep_blocked").sum()),
        "expected": 16,
        "status": "PASS" if int(leakage_decisions["human_decision"].astype(str).str.lower().eq("keep_blocked").sum()) == 16 else "FAIL",
    },
]
for target, expected in EXPECTED_TARGET_COUNTS.items():
    actual = int(count_map.get(target, -1))
    checks.append(
        {
            "check": f"strict_target_sample_count__{target}",
            "actual": actual,
            "expected": expected,
            "status": "PASS" if actual == expected else "FAIL",
        }
    )

check_df = pd.DataFrame(checks)
display(check_df)
if (check_df["status"] == "FAIL").any():
    raise RuntimeError("strict-clean/manual-approved contract check failed")


,check,actual,expected,status
0,strict_d08_shape,"(7592, 151)","(7592, 151)",PASS
1,strict_d08_sha256,5f56e375fd1c0474a5e55652859ae007e2f45becd6d335...,current strict-clean file hash,INFO
2,manual_approved_feature_registry_shape,"(198, 11)","(198, 11)",PASS
3,manual_approved_p4_use_true_n,131,131,PASS
4,manual_target_scope_policy_rows,32,32,PASS
5,human_required_decision_sheet_approved,13,13,PASS
6,target_leakage_override_sheet_keep_blocked,16,16,PASS
7,strict_target_sample_count__a_rate_pct,7592,7592,PASS
8,strict_target_sample_count__cd_rate_pct,7592,7592,PASS
9,strict_target_sample_count__f_rate_pct,7592,7592,PASS


## Gate 2 - Target Samples And Feature Policy


In [4]:
display(Markdown("### Target-specific sample counts"))
display(target_counts.sort_values("target_candidate"))

display(Markdown("### Manual-approved feature use"))
feature_summary = (
    manual_features.assign(manual_approved_p4_use=as_bool(manual_features["manual_approved_p4_use"]))
    .groupby(["manual_approved_p4_use", "p4_use", "strict_global_block_reason", "manual_review_scope_decision"], dropna=False)
    .size()
    .reset_index(name="columns")
    .sort_values(
        ["manual_approved_p4_use", "p4_use", "strict_global_block_reason", "manual_review_scope_decision"],
        ascending=[False, False, True, True],
    )
)
display(feature_summary)

display(Markdown("### Target-specific keep-blocked leakage policy"))
blocked = manual_policy[
    manual_policy["manual_decision"].astype(str).str.lower().eq("keep_blocked")
].copy()
display(blocked)

display(Markdown("### Strict target membership spot check"))
membership_with_school = target_membership.merge(
    d08[["outcome_row_id", "school_uid"]],
    on="outcome_row_id",
    how="left",
    validate="many_to_one",
)
membership_summary = (
    membership_with_school.assign(strict_target_keep=as_bool(membership_with_school["strict_target_keep"]))
    .groupby("target_candidate", dropna=False)
    .agg(rows=("outcome_row_id", "size"), keep_n=("strict_target_keep", "sum"), schools=("school_uid", "nunique"))
    .reset_index()
)
display(membership_summary)


### Target-specific sample counts

,target_candidate,strict_target_keep_n,total_n,common_structural_drop_n,target_missing_drop_n,target_extreme_small_denom_drop_n,strict_target_drop_n
0,a_rate_pct,7592,10242,2650,0,236,2650
1,cd_rate_pct,7592,10242,2650,0,297,2650
2,f_rate_pct,7592,10242,2650,0,683,2650
3,graduate_school_progression_rate_pct,5674,10242,2650,2655,346,4568
4,health_employment_rate_pct,5600,10242,2650,2765,190,4642
5,selectivity_proxy_pct,3119,10242,2650,6505,0,7123


### Manual-approved feature use

,manual_approved_p4_use,p4_use,strict_global_block_reason,manual_review_scope_decision,columns
5,True,True,NaN,inherit_strict_global_policy,131
4,False,True,self_target_leakage,inherit_strict_global_policy,2
0,False,False,all_null_source_unavailable,exclude_from_features_manual_all_null,4
1,False,False,job_cert_bridge_direct_join_prohibited,drop_D05_scope_manual_review,35
2,False,False,target_candidate_not_global_feature,inherit_strict_global_policy,6
3,False,False,NaN,inherit_strict_global_policy,20


### Target-specific keep-blocked leakage policy

,scope,target_or_artifact,manual_decision,approved_n,blocked_or_dropped_n,evidence,note
16,target_specific_feature_block,health_employment_rate_pct <- employment_rate_pct,keep_blocked,0,1,target_leakage_override_sheet.csv,same_target_family_outcome|employment_lineage_...
17,target_specific_feature_block,health_employment_rate_pct <- has_employment,keep_blocked,0,1,target_leakage_override_sheet.csv,employment_lineage_name
18,target_specific_feature_block,employment_rate_pct <- employment_rate_pct,keep_blocked,0,1,target_leakage_override_sheet.csv,same_as_target|corr_abs_gt_0.995|deterministic...
19,target_specific_feature_block,employment_rate_pct <- has_employment,keep_blocked,0,1,target_leakage_override_sheet.csv,employment_lineage_name
20,target_specific_feature_block,graduate_school_progression_rate_pct <- progre...,keep_blocked,0,1,target_leakage_override_sheet.csv,same_target_family_outcome|progression_lineage...
21,target_specific_feature_block,graduate_school_progression_rate_pct <- vocati...,keep_blocked,0,1,target_leakage_override_sheet.csv,same_target_family_outcome|progression_lineage...
22,target_specific_feature_block,graduate_school_progression_rate_pct <- univer...,keep_blocked,0,1,target_leakage_override_sheet.csv,same_target_family_outcome|progression_lineage...
23,target_specific_feature_block,graduate_school_progression_rate_pct <- domest...,keep_blocked,0,1,target_leakage_override_sheet.csv,same_target_family_outcome|progression_lineage...
24,target_specific_feature_block,graduate_school_progression_rate_pct <- overse...,keep_blocked,0,1,target_leakage_override_sheet.csv,same_target_family_outcome|progression_lineage...
25,target_specific_feature_block,graduate_school_progression_rate_pct <- has_pr...,keep_blocked,0,1,target_leakage_override_sheet.csv,progression_lineage_name


### Strict target membership spot check

,target_candidate,rows,keep_n,schools
0,a_rate_pct,10242,7592,197
1,cd_rate_pct,10242,7592,197
2,f_rate_pct,10242,7592,197
3,graduate_school_progression_rate_pct,10242,5674,197
4,health_employment_rate_pct,10242,5600,197
5,selectivity_proxy_pct,10242,3119,197


## Gate 3 - Readiness v4 Linkage


In [5]:
status_path = READINESS / "reports/P4_FINAL_MODELING_READINESS.json"
registry_path = READINESS / "artifacts/P4_PHASE_SAMPLE_REGISTRY_FINAL.csv"
feature_set_path = READINESS / "artifacts/P4_PHASE_FEATURE_SET_FINAL.csv"
denylist_path = READINESS / "qa/P4_TARGET_SPECIFIC_DENYLIST_FINAL.csv"
verification_path = READINESS / "qa/P4_STRICT_MANUAL_POLICY_VERIFICATION.csv"

if not status_path.exists():
    display(Markdown("P4 readiness v4 output is not present yet. Run `scripts/freeze_p2_g4_modeling_readiness_v4.py`."))
else:
    status = json.loads(status_path.read_text(encoding="utf-8"))
    display(
        pd.DataFrame(
            [
                {
                    "overall_status": status.get("overall_status"),
                    "active_d08_policy": status.get("active_d08_policy"),
                    "active_d08_shape": status.get("active_d08_shape"),
                    "active_d08_sha256": status.get("active_d08_sha256"),
                    "manual_approved_p4_use_true_n": status.get("manual_approved_p4_use_true_n"),
                    "target_specific_keep_blocked_n": status.get("target_specific_keep_blocked_n"),
                    "strict_deleted_rows_preserved_n": status.get("strict_deleted_rows_preserved_n"),
                }
            ]
        )
    )
    for title, path in [
        ("Sample registry", registry_path),
        ("Feature set registry", feature_set_path),
        ("Target-specific denylist", denylist_path),
        ("Strict/manual policy verification", verification_path),
    ]:
        display(Markdown(f"### {title}\n`{path.relative_to(ROOT)}`"))
        df = pd.read_csv(path)
        display(df.head(200))
        print("shape:", df.shape)


,overall_status,active_d08_policy,active_d08_shape,active_d08_sha256,manual_approved_p4_use_true_n,target_specific_keep_blocked_n,strict_deleted_rows_preserved_n
0,READY_WITH_WARNINGS,strict_clean_manual_approved,"[7592, 151]",5f56e375fd1c0474a5e55652859ae007e2f45becd6d335...,131,16,2650


### Sample registry
`workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_SAMPLE_REGISTRY_FINAL.csv`

,sample_id,row_n,school_n,campus_n,major7_n,train_n,validation_n,test_n,excluded_conflict_n,strict_clean_base_n,row_id_hash,status
0,P1_STRUCTURE_READY,5600,185,242,7,4080,871,649,0,7592,3f7f76bc4e1a9680052b996b012021d36ccf3374fa182f...,READY
1,P1_SELECTIVITY_READY,2355,130,159,7,1742,360,253,0,7592,6004094369f652f75ce961f0c9442a9598f5d80855150c...,READY
2,P2_STRUCTURE_READY,7592,197,261,7,5534,1168,890,0,7592,2e3fb7557ee894b50542591376211ec84817501cd7b412...,READY
3,P2_SELECTIVITY_READY,3119,146,179,7,2293,514,312,0,7592,1a0e36bb24d7e7f894897503bf0baf2c36f9b8710978b9...,READY
4,P3_STRUCTURE_READY,7592,197,261,7,5534,1168,890,0,7592,2e3fb7557ee894b50542591376211ec84817501cd7b412...,READY
5,P3_SELECTIVITY_READY,3119,146,179,7,2293,514,312,0,7592,1a0e36bb24d7e7f894897503bf0baf2c36f9b8710978b9...,READY
6,P4_E_STRUCTURE_READY,5600,185,242,7,4080,871,649,0,7592,3f7f76bc4e1a9680052b996b012021d36ccf3374fa182f...,READY
7,P4_P_STRUCTURE_READY,5674,194,254,7,4129,884,661,0,7592,6625f4fcb391ac2e7d9a3001be72425a2dd4f0f6e03ef4...,READY
8,P4_JOINT_STRUCTURE_READY,5600,185,242,7,4080,871,649,0,7592,3f7f76bc4e1a9680052b996b012021d36ccf3374fa182f...,READY
9,P4_E_SELECTIVITY_READY,2355,130,159,7,1742,360,253,0,7592,6004094369f652f75ce961f0c9442a9598f5d80855150c...,READY


shape: (12, 12)


### Feature set registry
`workbook/p2/p2_4/p4_modeling_readiness_v4/artifacts/P4_PHASE_FEATURE_SET_FINAL.csv`

...

,target,blocked_feature,policy
0,employment_rate_pct,employment_rate_pct,keep_blocked
1,employment_rate_pct,has_employment,keep_blocked
2,graduate_school_progression_rate_pct,domestic_progression_rate_pct,keep_blocked
3,graduate_school_progression_rate_pct,has_progression,keep_blocked
4,graduate_school_progression_rate_pct,overseas_progression_rate_pct,keep_blocked
5,graduate_school_progression_rate_pct,progression_rate_pct,keep_blocked
6,graduate_school_progression_rate_pct,university_progression_rate_pct,keep_blocked
7,graduate_school_progression_rate_pct,vocational_college_progression_rate_pct,keep_blocked
8,health_employment_rate_pct,employment_rate_pct,keep_blocked
9,health_employment_rate_pct,has_employment,keep_blocked


shape: (16, 3)


### Strict/manual policy verification
`workbook/p2/p2_4/p4_modeling_readiness_v4/qa/P4_STRICT_MANUAL_POLICY_VERIFICATION.csv`

,check,actual,expected,status,path
0,strict_d08_shape,"[7592, 151]","[7592, 151]",PASS,NaN
1,manual_approved_p4_use_true_n,131,131,PASS,NaN
2,strict_target_sample_count__a_rate_pct,7592,7592,PASS,NaN
3,strict_target_sample_count__cd_rate_pct,7592,7592,PASS,NaN
4,strict_target_sample_count__f_rate_pct,7592,7592,PASS,NaN
5,strict_target_sample_count__graduate_school_pr...,5674,5674,PASS,NaN
6,strict_target_sample_count__health_employment_...,5600,5600,PASS,NaN
7,strict_target_sample_count__selectivity_proxy_pct,3119,3119,PASS,NaN
8,manual_target_scope_policy_rows,32,32,PASS,NaN
9,human_required_decision_sheet_approved,13,13,PASS,workbook/p2/p2_4/source_eda/strict_clean_v1/hu...


shape: (11, 5)


## Final Contract Status


In [6]:
final_status = {
    "P2_G4_1_STRICT_CLEAN_INPUT_STATUS": "READY",
    "P2_G4_1_MANUAL_APPROVED_POLICY_STATUS": "READY",
    "P2_G4_1_TARGET_SAMPLE_STATUS": "READY",
    "P2_G4_1_TARGET_LEAKAGE_STATUS": "KEEP_BLOCKED_TARGET_SPECIFIC",
    "P2_G4_1_PANEL_STATUS": "NOT_AVAILABLE_BY_POLICY",
}
display(pd.DataFrame([final_status]))


,P2_G4_1_STRICT_CLEAN_INPUT_STATUS,P2_G4_1_MANUAL_APPROVED_POLICY_STATUS,P2_G4_1_TARGET_SAMPLE_STATUS,P2_G4_1_TARGET_LEAKAGE_STATUS,P2_G4_1_PANEL_STATUS
0,READY,READY,READY,KEEP_BLOCKED_TARGET_SPECIFIC,NOT_AVAILABLE_BY_POLICY
